# Free AI Video Batch Maker (LTX-Video 2B on Colab)

Type your prompts, run all cells, and get one AI video per prompt. No API key, no payment, 100% free using Google Colab's free GPU and the open-source LTX-Video 2B model by Lightricks.

## Setup (do this once)

1. Open this notebook in Google Colab (File > Open notebook > GitHub tab > paste the repo URL).
2. Go to **Runtime > Change runtime type** and set Hardware accelerator to **T4 GPU**, then Save.
3. Run the cells below **top to bottom** (Runtime > Run all works too).
4. When finished, the last cell downloads all your videos as one ZIP file.

In [ ]:
!nvidia-smi

In [ ]:
# Install everything needed (takes a few minutes the first time)
!pip install -q torch diffusers accelerate transformers safetensors imageio imageio-ffmpeg

# NOTE: if Colab asks you to restart the runtime after install, click Restart, then skip this cell on the next run.

In [ ]:
import torch
from diffusers import LTXPipeline
from diffusers.utils import export_to_video
from transformers import T5EncoderModel, T5Tokenizer

MODEL_URL = "https://huggingface.co/Lightricks/LTX-Video/ltx-video-2b-v0.9.1.safetensors"

print("Loading text encoder...")
text_encoder = T5EncoderModel.from_pretrained(
    "Lightricks/LTX-Video", subfolder="text_encoder", torch_dtype=torch.bfloat16)
tokenizer = T5Tokenizer.from_pretrained(
    "Lightricks/LTX-Video", subfolder="tokenizer")

print("Loading model (first run downloads about 4 GB, then it is cached)...")
pipe = LTXPipeline.from_single_file(
    MODEL_URL, text_encoder=text_encoder, tokenizer=tokenizer,
    torch_dtype=torch.bfloat16)
# Aggressive memory saving for free T4: only one component on GPU at a time,
# plus tiled VAE decoding. Slower, but fits in 16GB VRAM.
pipe.enable_sequential_cpu_offload()
pipe.vae.enable_tiling()
print("Model loaded")

## EDIT THIS CELL: put your prompts here

One line per video. Write what you want to see, keep it visual and specific. Portrait 9:16 is used so the videos fit Shorts, Reels and TikTok.

In [ ]:
# EDIT ME: one prompt per line = one video
prompts = [
    "A happy 7-year-old boy in a red football jersey kicking a soccer ball across a sunny green field, other kids playing football in the background, joyful energy, bright daylight, cinematic sports video",
    "The same young boy in a red football jersey sitting sadly on the grass after losing a football match, teammates walking away disappointed, quiet stadium mood, soft evening light, emotional cinematic video",
    "The sad little boy in a red football jersey running into his mother's arms crying after losing the match, she kneels down and hugs him warmly to comfort him, golden hour light in a park, heartwarming emotional video",
]

import os
os.makedirs("/content/output", exist_ok=True)

for i, prompt in enumerate(prompts, start=1):
    out_path = f"/content/output/video-{i:03d}.mp4"
    if os.path.exists(out_path):
        print(f"[{i}/{len(prompts)}] skipped (already exists): {out_path}")
        continue
    print(f"[{i}/{len(prompts)}] generating: {prompt[:70]}...")
    try:
        frames = pipe(
            prompt=prompt,
            negative_prompt="blurry, low quality, distorted, watermark, text",
            width=448,
            height=768,
            num_frames=65,
            num_inference_steps=25,
        ).frames[0]
        export_to_video(frames, out_path, fps=24)
        print(f"[{i}/{len(prompts)}] saved: {out_path}")
    except Exception as e:
        print(f"[{i}/{len(prompts)}] FAILED: {e}")
        continue

print("Done.")

In [ ]:
# Zip all videos and download them
!zip -qr /content/videos.zip /content/output
from google.colab import files
files.download("/content/videos.zip")
print("Download started: videos.zip")

**Honest limits**
- Free Colab can disconnect; keep batches to 3-5 prompts per run.
- Memory-saving mode is ON so the model fits on a free T4: each clip takes
  roughly 10-20 minutes. Start it and let it run.
- Clips are about 2.5 seconds, 448x768 portrait (9:16-ish, good for
  Shorts/Reels/TikTok). Quality is good for an open model, not Veo-level.